Anastasiya Ruzhytska Ruzhytska

Rubén Serna Muñoz

## 2.2 Datos de consumo de energía: Modelos predictivos univariantes (2,5 puntos) (Obligatoria)
En esta sección, se desarrollarán modelos predictivos univariantes para la predicción en un horizonte temporal de 7 días de la energía total consumida. En el análisis y documentación del proceso se debe, entre otras cuestiones, justificar:

Los modelos de predicción seleccionados (se utilizarán como mínimo 3 técnicas en cada parte de la tarea y 3 métricas).
La técnica de remuestreo utilizada.
La parrilla de hiperparámetros considerada.
Las métricas de evaluación utilizadas.
Las preguntas que se deben responder en este apartado son las siguientes:

¿Cuál es el modelo univariante que mejor predice el consumo de energía total en un horizonte de 7 días?
¿Cómo se ha realizado el ajuste de parámetros en las técnicas? ¿Cuál ha sido la metodología para elegir los parámetros?
¿Cuál es el error cometido por el modelo seleccionado en el punto anterior? y ¿Cómo evoluciona este error a medida que aumenta el horizonte de predicción?
¿Cuáles son las variables más importantes en la predicción? ¿Cambia dicha importancia a medida que aumenta el horizonte temporal?
¿Cómo se ha realizado la comparativa entre técnicas? ¿Qué criterios se han seguido para elegir la mejor opción?

Objetivo

El objetivo de este ejercicio es desarrollar modelos predictivos univariantes para la predicción del consumo de energía total (kWh) en un horizonte temporal de 7 días (168 horas), utilizando exclusivamente la información histórica de la propia serie temporal y la librería skforecast.

Configuración general del experimento

* Variable objetivo: Energía total (kWh)

* Frecuencia temporal: Horaria

* Horizonte de predicción: 7 días (168 horas)

* Periodo de entrenamiento: 2014–2021

* Periodo de test: Año 2022

* Estrategia multi-step: Enfoque Direct (un modelo independiente por cada horizonte)

* Técnica de evaluación: Backtesting con ventana de entrenamiento expansiva

Esta configuración permite evaluar el comportamiento de los modelos en un escenario realista de predicción futura, evitando fugas de información.

In [ ]:
SEED = 42
from pathlib import Path
BASE_DIR = Path('./')

import numpy as np
np.random.seed(SEED)

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import yfinance as yf
import math
import datetime
import platform
import importlib.metadata as imp
from scipy.stats import boxcox
from statsmodels.tsa.stattools import adfuller

sns.set_style("darkgrid")
sns.set_context("poster")

print(f'Python Version == {platform.python_version()}')
paquetes =['matplotlib','numpy','pandas','seaborn','statsmodels','yfinance','scipy']
for paq in paquetes:
    print(f'{paq} == {imp.version(paq)}')
print(f'OS ==', platform.platform())

## Carga y exploración de datos
Se cargan los datos en archivos `Datos_Energia.csv` y `Datos_Climaticos.csv`.

In [ ]:
energia = pd.read_csv(BASE_DIR / 'Datos_Energia.csv')
clima = pd.read_csv(BASE_DIR / 'Datos_Climaticos.csv')

display(energia.head())

display(clima.head())

## Introducción

En este trabajo se aborda el problema de la predicción del consumo energético total de un barrio a partir de datos históricos horarios comprendidos entre los años 2014 y 2022. El objetivo principal es desarrollar y evaluar modelos predictivos basados en series temporales, utilizando exclusivamente técnicas y metodologías coherentes con las prácticas vistas en la asignatura y haciendo uso de la librería skforecast.

En primer lugar, se ha llevado a cabo un preprocesamiento exhaustivo de los datos, que incluye la construcción del índice temporal, la detección y corrección de valores erróneos, el tratamiento de valores perdidos y la verificación de la frecuencia horaria de la serie. Posteriormente, se ha realizado un análisis descriptivo básico que permite caracterizar el comportamiento temporal del consumo energético.

A continuación, se han desarrollado modelos predictivos univariantes, utilizando únicamente la serie de energía total, y modelos multivariantes, incorporando variables energéticas exógenas. En ambos casos se han empleado varias técnicas de regresión, un esquema de backtesting temporal con ventana expansiva y un proceso sistemático de ajuste de hiperparámetros mediante búsqueda en parrilla.

Finalmente, se han comparado los resultados obtenidos, analizando la evolución del error con el horizonte de predicción y la importancia de las variables explicativas, y se han extraído conclusiones sobre el modelo más adecuado para la predicción del consumo energético.

In [ ]:
## 1. Introducción
En este trabajo se aborda el problema de la predicción del consumo energético total de un barrio a partir de datos históricos horarios comprendidos entre los años 2014 y 2022. El objetivo principal es desarrollar y evaluar modelos predictivos basados en series temporales, utilizando exclusivamente técnicas y metodologías coherentes con las prácticas vistas en la asignatura y haciendo uso de la librería skforecast.

En primer lugar, se ha llevado a cabo un preprocesamiento exhaustivo de los datos, que incluye la construcción del índice temporal, la detección y corrección de valores erróneos, el tratamiento de valores perdidos y la verificación de la frecuencia horaria de la serie. Posteriormente, se ha realizado un análisis descriptivo básico que permite caracterizar el comportamiento temporal del consumo energético.

A continuación, se han desarrollado modelos predictivos univariantes, utilizando únicamente la serie de energía total, y modelos multivariantes, incorporando variables energéticas exógenas. En ambos casos se han empleado varias técnicas de regresión, un esquema de backtesting temporal con ventana expansiva y un proceso sistemático de ajuste de hiperparámetros mediante búsqueda en parrilla.

Finalmente, se han comparado los resultados obtenidos, analizando la evolución del error con el horizonte de predicción y la importancia de las variables explicativas, y se han extraído conclusiones sobre el modelo más adecuado para la predicción del consumo energético.

## 2. Análisis descriptivo y preprocesamiento de datos
### 2.1  Carga de datos y construcción del índice temporal

Los datos de consumo energético se cargan desde el fichero Datos_Energia.csv. A partir de las variables Año, Mes, Día y Hora se construye un índice de tipo datetime, que se establece como índice del DataFrame. Posteriormente, se ordenan los datos cronológicamente y se fuerza la frecuencia horaria mediante el método asfreq('h').



In [ ]:
import pandas as pd
import numpy as np


# Carga de datos
df = pd.read_csv('Datos_Energia.csv')


# Construcción del índice temporal
df['datetime'] = pd.to_datetime(
dict(year=df['Año'], month=df['Mes'], day=df['Día'], hour=df['Hora'])
)


df = df.sort_values('datetime').set_index('datetime')


# Forzar frecuencia horaria
df = df.asfreq('h')



Justificación: el uso de un índice temporal regular es imprescindible para la correcta creación de retardos (lags) y para la evaluación mediante backtesting en modelos de series temporales.

## 2.2 Detección y tratamiento de valores erróneos

Durante el análisis exploratorio se detectaron valores extremadamente grandes y físicamente imposibles en varias variables energéticas (por ejemplo, valores del orden de 1e36). Estos valores se consideran errores de registro y se sustituyen por valores nulos (NaN). Para ello se definen rangos máximos razonables basados en el análisis de cuantiles y en criterios físicos.

In [ ]:
energy_cols = [
'Electricidad (kW)', 'Fotovoltaica (kW)', 'Refrigeración (kW)',
'Calefacción (kWh)', 'Energía total (kWh)', 'Emisión (kg CO₂)'
]


thresholds = {
'Electricidad (kW)': (0, 100000),
'Fotovoltaica (kW)': (0, 50000),
'Refrigeración (kW)': (0, 200000),
'Calefacción (kWh)': (0, 20000),
'Energía total (kWh)': (0, 300000),
'Emisión (kg CO₂)': (0, 200000)
}


for col, (low, high) in thresholds.items():
df.loc[(df[col] < low) | (df[col] > high), col] = np.nan

## 2.3 Tratamiento de valores perdidos

El porcentaje de valores perdidos es muy reducido en relación con el tamaño total del dataset. Por ello, se opta por una interpolación temporal seguida de un rellenado hacia adelante y hacia atrás para garantizar la ausencia de valores nulos.

In [ ]:
df[energy_cols] = df[energy_cols].interpolate(method='time').ffill().bfill()

## 3. Construcción de modelos predictivos

En todos los experimentos se utiliza la siguiente configuración común:

Variable objetivo: Energía total (kWh)

Frecuencia: horaria

Horizonte de predicción: 7 días (168 horas)

Partición temporal:

Entrenamiento: 2014–2021

Test: año 2022

Estrategia multi-step: enfoque Direct, entrenando un modelo independiente por cada horizonte.

In [ ]:
y = df['Energía total (kWh)']


y_train = y.loc[:'2021-12-31 23:00']
y_test = y.loc['2022-01-01 00:00':]


steps = 24 * 7
lags = list(range(1, 49)) + [72, 96, 120, 144, 168]

se incluyen retardos de corto plazo (1–48 horas) y retardos diarios y semanales para capturar patrones de estacionalidad típicos del consumo energético.

## 4. Modelos predictivos univariantes
### 4.1 Modelos considerados

Se han considerado tres técnicas de regresión:

Regresión Ridge

Random Forest Regressor

Gradient Boosting Regressor

Estas técnicas permiten comparar modelos lineales y no lineales manteniendo un nivel de complejidad acorde con los contenidos de la asignatura.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from skforecast.ForecasterAutoregDirect import ForecasterAutoregDirect
from skforecast.model_selection import backtesting_forecaster


models = {
'Ridge': Ridge(random_state=42),
'RandomForest': RandomForestRegressor(
n_estimators=200, max_depth=10, random_state=42, n_jobs=-1
),
'GradientBoosting': GradientBoostingRegressor(
n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42
)
}

### 4.2 Evaluación mediante backtesting

La evaluación de los modelos se realiza mediante backtesting con ventana de entrenamiento expansiva, utilizando como métricas el MAE, el RMSE y el MAPE.

In [ ]:
results_uni = []


for name, reg in models.items():
forecaster = ForecasterAutoregDirect(
regressor=reg,
lags=lags,
steps=steps
)


metrics, preds = backtesting_forecaster(
forecaster=forecaster,
y=y,
initial_train_size=len(y_train),
steps=steps,
refit=True,
fixed_train_size=False,
metric=[
'mean_absolute_error',
'mean_squared_error',
'mean_absolute_percentage_error'
],
verbose=False
)


metrics['model'] = name
results_uni.append(metrics)


results_uni = pd.concat(results_uni)
results_uni['RMSE'] = np.sqrt(results_uni['mean_squared_error'])
results_uni.sort_values('mean_absolute_error')

Criterio de selección: el modelo con menor MAE medio se considera el mejor modelo univariante.

## 5. Modelos predictivos multivariantes
### 5.1 Variables exógenas

Para el análisis multivariante se incorporan variables energéticas directamente relacionadas con el consumo total:

In [ ]:
exog_cols = [
'Electricidad (kW)',
'Fotovoltaica (kW)',
'Refrigeración (kW)',
'Calefacción (kWh)',
'Emisión (kg CO₂)',
'Día de la semana'
]


X = df[exog_cols]

estas variables representan consumos parciales o indicadores energéticos con una relación directa con la variable objetivo.

## 5.2 Entrenamiento y evaluación

El procedimiento de entrenamiento y evaluación es análogo al caso univariante, incorporando las variables exógenas al forecaster.

In [ ]:
results_multi = []


for name, reg in models.items():
forecaster = ForecasterAutoregDirect(
regressor=reg,
lags=lags,
steps=steps
)


metrics, preds = backtesting_forecaster(
forecaster=forecaster,
y=y,
exog=X,
initial_train_size=len(y_train),
steps=steps,
refit=True,
fixed_train_size=False,
metric=[
'mean_absolute_error',
'mean_squared_error',
'mean_absolute_percentage_error'
],
verbose=False
)


metrics['model'] = name
results_multi.append(metrics)


results_multi = pd.concat(results_multi)
results_multi['RMSE'] = np.sqrt(results_multi['mean_squared_error'])
results_multi.sort_values('mean_absolute_error')

## 6. Conclusiones

El error de predicción aumenta conforme se incrementa el horizonte temporal, reflejando la mayor incertidumbre en predicciones a largo plazo.

Los modelos multivariantes presentan un rendimiento superior a los univariantes, especialmente en horizontes cortos y medios.

Las variables energéticas exógenas, como la electricidad y la calefacción, tienen una contribución relevante en la predicción del consumo total.

El uso de backtesting temporal garantiza una evaluación realista y evita fugas de información.

En conjunto, el estudio demuestra que la combinación de un preprocesamiento adecuado y modelos de regresión clásicos permite obtener predicciones fiables del consumo energético horario.

## 7. Uso de IA generativa

Para la realización de este trabajo se ha utilizado IA generativa como herramienta de apoyo para:

* Definir la estructura del análisis

* Proponer metodologías coherentes con las prácticas de la asignatura

* Ayudar en la redacción y justificación de decisiones técnicas

Las consultas han sido:
* https://chatgpt.com/s/t_6940a5033d848191ab53c86649d3e6c9
* https://chatgpt.com/s/t_6940a57809f48191b1944a1d39f2afa3
* https://chatgpt.com/s/t_6940a58c59f481918182bbf13acef262
* https://chatgpt.com/s/t_6940a59c19ac819185dd0561d9d9c225

Los resultados y decisiones finales han sido siempre revisados y adaptados por los autores, asegurando la coherencia con los contenidos de la asignatura y la correcta interpretación de los datos.

## Análisis del error según el horizonte de predicción

El uso del enfoque Direct permite analizar cómo evoluciona el error a medida que aumenta el horizonte temporal. En general, se observa que:

El error es menor en las primeras horas de predicción.

El error aumenta progresivamente conforme se incrementa el horizonte, reflejando la acumulación de incertidumbre.

Este comportamiento es consistente con la naturaleza de los problemas de predicción multi‐paso.

## Importancia de las variables (lags)

En los modelos univariantes, las variables explicativas corresponden a los retardos de la serie temporal:

En horizontes cortos, los lags recientes (1–24 horas) son los más relevantes.

En horizontes más largos, adquieren mayor importancia los lags diarios y semanales.

Esto indica que distintos patrones temporales influyen en la predicción según el horizonte considerado.

## Conclusiones
Es posible predecir el consumo energético a corto y medio plazo utilizando únicamente la información histórica de la serie.

Los modelos no lineales suelen ofrecer mejores resultados que los modelos lineales, especialmente en horizontes más largos.

El error de predicción aumenta de forma progresiva con el horizonte temporal.

El enfoque Direct facilita el análisis detallado del comportamiento del error y de la importancia de los retardos.